In [22]:
import torch
import torch.nn as nn
import torch.nn.init as init
import torchvision
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader, random_split


# Готовим данные
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


# Подготовка данных из прошлого урока
dataset = MNIST(root='./data', train=True, download=True, transform=transform)
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size

train_data, val_data = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False)
images, labels = next(iter(train_loader))


# Определяем модель через nn.Sequential
model_seq = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

# Прогон и обучение одного батча
out_seq = model_seq(images)
print("Output shape:", out_seq.shape)   # ожидаем [64, 10]

# Оптимизатор и функция потерь
optimizer = torch.optim.SGD(model_seq.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Прямой проход
optimizer.zero_grad()
loss = criterion(out_seq, labels)
loss.backward()
optimizer.step()

print("Loss после одного шага:", loss.item())

Output shape: torch.Size([64, 10])
Loss после одного шага: 2.3440616130828857


# AdvancedMNISTMLP

In [26]:
# Реализация класса
class AdvancedMNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.linear2 = nn.Linear(256, 128)
        self.linear3 = nn.Linear(128, 10)

        # Инициализация Xavier
        for layer in (self.linear, self.linear2, self.linear3):
            init.xavier_uniform_(layer.weight)
            layer.bias.data.zero_()

    def forward(self, x):
        # Разворачиваем изображение (batch,1,28,28) → (batch,784)
        x = x.view(x.size(0), -1)
        x = self.linear(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.relu(x)
        x = self.linear3(x)
        return x


# Загрузка данных и DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)


# Один батч и прямой проход
model = AdvancedMNISTMLP()


images, labels = next(iter(train_loader))


outputs = model(images)


# Вывод результатов
print("Output shape:", outputs.shape)      # ожидаем [64, 10]
print("Batch labels shape:", labels.shape) # [64]

Output shape: torch.Size([64, 10])
Batch labels shape: torch.Size([64])


# AdvancedMNISTMLP nn.Sequential

In [30]:

class AdvancedMNISTMLPSeq(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                m.bias.data.zero_()

    def forward(self, x):
        return self.net(x)

# Загрузка данных и DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Один батч и прямой проход
model = AdvancedMNISTMLPSeq()

images, labels = next(iter(train_loader))

outputs = model(images)

# Вывод результатов
print("Output shape:", outputs.shape)      # ожидаем [64, 10]
print("Batch labels shape:", labels.shape) # [64]

Output shape: torch.Size([64, 10])
Batch labels shape: torch.Size([64])


# FlexibleMNISTMLP. Using ModuleList

In [ ]:
class FlexibleMNISTMLP(nn.Module):
    def __init__(self, hidden_sizes=(256, 128), dropout_p=0.5):
        super().__init__()
        self.flatten = nn.Flatten()

        # Собираем скрытые слои динамически
        layers = []
        in_dim = 784
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_p))
            in_dim = h  # выход текущего слоя = вход следующего

        self.hidden = nn.ModuleList(layers)
        self.output = nn.Linear(in_dim, 10)

        # Инициализация Xavier для всех Linear-слоёв
        for m in self.hidden:
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                m.bias.data.zero_()
        init.xavier_uniform_(self.output.weight)
        self.output.bias.data.zero_()

    def forward(self, x):
        x = self.flatten(x)
        for layer in self.hidden:
            x = layer(x)

        x = self.output(x)
        return x


# Загрузка данных
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# модель с архитектурой [256, 128]
model = FlexibleMNISTMLP(hidden_sizes=[256, 128], dropout_p=0.5)

images, labels = next(iter(train_loader))
outputs = model(images)

print("Output shape:", outputs.shape)       # ожидаем [64, 10]
print("Batch labels shape:", labels.shape)   # [64]
print("Число скрытых модулей:", len(model.hidden))  # 6 (Linear+ReLU+Dropout × 2)

model_deep = FlexibleMNISTMLP(hidden_sizes=[512, 256, 128, 64])
outputs_deep = model_deep(images)
print("\nГлубокая модель — Output shape:", outputs_deep.shape)  # [64, 10]
print("Число скрытых модулей:", len(model_deep.hidden))         # 12